# Model selection spike, run 5: Qwen3-8B (thinking off) and Llama-3.1-8B-Instruct

**Why run 5:** in run 4, Qwen3-8B stayed in its "thinking" mode and used all 300 tokens before giving an answer, so it could not be scored; and the GPU build was not used, so the run was slow. Llama-3.1-8B-Instruct passed the most questions in run 4.

**Changes from run 4:**
1. **Qwen3 thinking is switched off** the way its chat template does it (an empty `<think></think>` block at the start of the answer), not only with `/no_think`.
2. **GPU check:** the install cell tries several prebuilt CUDA wheels and then checks that llama.cpp can really use the GPU; if not, it builds with CUDA (about 20 minutes) instead of silently running on CPU.
3. Answers that still contain thinking text fail the scoring.

Same as run 4: questions, provisions, prompt, rule checks, CPU speed measurement on one question. Phi-4-mini can be added with `COMPARE_PHI = True` (its answers do not change between runs, temperature 0).

**How to run (Türkçe):** Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**. İlk hücre sonunda "GPU offload: True" yazmalı. Sonunda `spike_results_run5.csv` ve `spike_cpu_speed_run5.csv` iner.

In [ ]:
# 1. Install llama-cpp-python with GPU support and check that the GPU is really used
import subprocess, shutil, sys
GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU visible:", GPU)

def gpu_offload_ok():
    r = subprocess.run([sys.executable, "-c", "import llama_cpp; print(llama_cpp.llama_supports_gpu_offload())"], capture_output=True, text=True)
    return r.stdout.strip() == "True"

if GPU:
    for cu in ["cu124", "cu123", "cu122", "cu121"]:
        subprocess.run(f"pip -q install --force-reinstall --no-deps --prefer-binary llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{cu}", shell=True)
        subprocess.run("pip -q install llama-cpp-python", shell=True)   # dependencies
        if gpu_offload_ok():
            print("Prebuilt wheel works:", cu); break
    else:
        print("No prebuilt CUDA wheel worked; building with CUDA (about 20 minutes) ...")
        subprocess.run('CMAKE_ARGS="-DGGML_CUDA=on" pip -q install --force-reinstall --no-cache-dir --no-binary llama-cpp-python llama-cpp-python', shell=True)
else:
    subprocess.run("pip -q install --prefer-binary llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu", shell=True)
subprocess.run("pip -q install huggingface_hub pandas", shell=True)
OFFLOAD = gpu_offload_ok()
print("GPU offload:", OFFLOAD)
if GPU and not OFFLOAD:
    print("WARNING: GPU is visible but llama.cpp cannot use it; the run will be slow (CPU).")

In [ ]:
# 2. Models
COMPARE_PHI = False
MODELS = {
    "Qwen3-8B":              ["Qwen/Qwen3-8B-GGUF", "unsloth/Qwen3-8B-GGUF"],
    "Llama-3.1-8B-Instruct": ["bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", "unsloth/Llama-3.1-8B-Instruct-GGUF"],
}
if COMPARE_PHI:
    MODELS["Phi-4-mini-instruct"] = ["unsloth/Phi-4-mini-instruct-GGUF", "bartowski/microsoft_Phi-4-mini-instruct-GGUF"]
QUANT = "*Q4_K_M.gguf"
SETTINGS = dict(temperature=0.0, max_tokens=300)
N_CTX = 4096
MEASURE_CPU = True

In [ ]:
DATA = {
 "passages": {
  "Annex III, point 3(a)": "AI systems intended to be used to determine access or admission or to assign natural persons to educational and vocational training institutions at all levels;",
  "Annex III, point 3(b)": "AI systems intended to be used to evaluate learning outcomes, including when those outcomes are used to steer the learning process of natural persons in educational and vocational training institutions at all levels;",
  "Article 5(1)(f)": "The following AI practices shall be prohibited: ... (f) the placing on the market, the putting into service for this specific purpose, or the use of AI systems to infer emotions of a natural person in the areas of workplace and education institutions, except where the use of the AI system is intended to be put in place or into the market for medical or safety reasons;",
  "Article 6(2)": "In addition to the high-risk AI systems referred to in paragraph 1, AI systems referred to in Annex III shall be considered to be high-risk.",
  "Article 6(3)": "By derogation from paragraph 2, an AI system referred to in Annex III shall not be considered to be high-risk where it does not pose a significant risk of harm to the health, safety or fundamental rights of natural persons, including by not materially influencing the outcome of decision making. The first subparagraph shall apply where any of the following conditions is fulfilled: (a) the AI system is intended to perform a narrow procedural task; (b) the AI system is intended to improve the result of a previously completed human activity; (c) the AI system is intended to detect decision-making patterns or deviations from prior decision-making patterns and is not meant to replace or influence the previously completed human assessment, without proper human review; or (d) the AI system is intended to perform a preparatory task to an assessment relevant for the purposes of the use cases listed in Annex III. Notwithstanding the first subparagraph, an AI system referred to in Annex III shall always be considered to be high-risk where the AI system performs profiling of natural persons.",
  "Article 6(4)": "A provider who considers that an AI system referred to in Annex III is not high-risk shall document its assessment before that system is placed on the market or put into service. Such provider shall be subject to the registration obligation set out in Article 49(2). Upon request of national competent authorities, the provider shall provide the documentation of the assessment.",
  "Article 49(2)": "Before placing on the market or putting into service an AI system for which the provider has concluded that it is not high-risk according to Article 6(3), that provider or, where applicable, the authorised representative shall register themselves and that system in the EU database referred to in Article 71.",
  "Article 15(1)": "High-risk AI systems shall be designed and developed in such a way that they achieve an appropriate level of accuracy, robustness, and cybersecurity, and that they perform consistently in those respects throughout their lifecycle.",
  "Article 15(3)": "The levels of accuracy and the relevant accuracy metrics of high-risk AI systems shall be declared in the accompanying instructions of use.",
  "Article 50(1)": "Providers shall ensure that AI systems intended to interact directly with natural persons are designed and developed in such a way that the natural persons concerned are informed that they are interacting with an AI system, unless this is obvious from the point of view of a natural person who is reasonably well-informed, observant and circumspect, taking into account the circumstances and the context of use.",
  "Article 113, third paragraph, point (c)(i)": "Chapter III, Sections 1, 2, and 3, with the exception of Article 6(5), shall apply from: (i) 2 December 2027 as regards AI systems classified as high-risk pursuant to Article 6(2) and Annex III;",
  "Article 14(4)(d)": "to decide, in any particular situation, not to use the high-risk AI system or to otherwise disregard, override or reverse the output of the high-risk AI system;",
  "Article 8(1)": "High-risk AI systems shall comply with the requirements laid down in this Section, taking into account their intended purpose as well as the generally acknowledged state of the art on AI and AI-related technologies. ...",
  "Article 9(1)": "A risk management system shall be established, implemented, documented and maintained in relation to high-risk AI systems.",
  "Article 10(1)": "High-risk AI systems which make use of techniques involving the training of AI models with data shall be developed on the basis of training, validation and testing data sets that meet the quality criteria ...",
  "Article 11(1)": "The technical documentation of a high-risk AI system shall be drawn up before that system is placed on the market or put into service and shall be kept up-to date.",
  "Article 12(1)": "High-risk AI systems shall technically allow for the automatic recording of events (logs) over the lifetime of the system.",
  "Article 13(1)": "High-risk AI systems shall be designed and developed in such a way as to ensure that their operation is sufficiently transparent to enable deployers to interpret a system's output and use it appropriately. ...",
  "Article 14(1)": "High-risk AI systems shall be designed and developed in such a way, including with appropriate human-machine interface tools, that they can be effectively overseen by natural persons during the period in which they are in use.",
  "Article 6(3), first subparagraph": "By derogation from paragraph 2, an AI system referred to in Annex III shall not be considered to be high-risk where it does not pose a significant risk of harm to the health, safety or fundamental rights of natural persons, including by not materially influencing the outcome of decision making. The first subparagraph shall apply where any of the following conditions is fulfilled:",
  "Article 6(3)(a)": "the AI system is intended to perform a narrow procedural task;",
  "Article 6(3)(b)": "the AI system is intended to improve the result of a previously completed human activity;",
  "Article 6(3)(c)": "the AI system is intended to detect decision-making patterns or deviations from prior decision-making patterns and is not meant to replace or influence the previously completed human assessment, without proper human review; or",
  "Article 6(3)(d)": "the AI system is intended to perform a preparatory task to an assessment relevant for the purposes of the use cases listed in Annex III.",
  "Article 6(3), last subparagraph": "Notwithstanding the first subparagraph, an AI system referred to in Annex III shall always be considered to be high-risk where the AI system performs profiling of natural persons."
 },
 "cases": [
  {
   "id": "B-01",
   "kind": "scenario",
   "ctx": [
    "Article 6(2)",
    "Article 6(3), first subparagraph",
    "Article 6(3)(a)",
    "Article 6(3)(b)",
    "Article 6(3)(c)",
    "Article 6(3)(d)",
    "Article 6(3), last subparagraph",
    "Annex III, point 3(a)",
    "Article 50(1)"
   ],
   "expect": [
    "6(3)"
   ],
   "question": "We made a tool that only takes the grades teachers already gave and calculates the weighted average. Is this still a high-risk AI system?",
   "expect_point": "6(3)(a)"
  },
  {
   "id": "B-02",
   "kind": "scenario",
   "ctx": [
    "Article 6(3), first subparagraph",
    "Article 6(3)(a)",
    "Article 6(3)(b)",
    "Article 6(3)(c)",
    "Article 6(3)(d)",
    "Article 6(3), last subparagraph",
    "Article 6(4)",
    "Article 50(1)",
    "Article 113, third paragraph, point (c)(i)"
   ],
   "expect": [
    "6(3)"
   ],
   "question": "Our AI reads the exam a teacher prepared and points out mistakes, like typos or unclear questions. It does not change the questions or make them easier or harder. Is this a problem under the AI Act?",
   "expect_point": "6(3)(b)"
  },
  {
   "id": "B-03",
   "kind": "scenario",
   "ctx": [
    "Article 6(4)",
    "Article 49(2)",
    "Article 14(4)(d)",
    "Annex III, point 3(a)"
   ],
   "expect": [
    "6(4)",
    "49(2)"
   ],
   "question": "Okay, and if we decide our grade calculator is not high-risk, do we still have to do anything before we launch it?",
   "prev_question": "We made a tool that only takes the grades teachers already gave and calculates the weighted average. Is this still a high-risk AI system?"
  },
  {
   "id": "B-04",
   "kind": "scenario",
   "ctx": [
    "Article 5(1)(f)",
    "Annex III, point 3(b)",
    "Article 50(1)"
   ],
   "expect": [
    "5(1)"
   ],
   "question": "We want to add a webcam feature that notices when students look bored or frustrated, so the lesson can change. Are we allowed to do this?"
  },
  {
   "id": "B-05",
   "kind": "scenario",
   "ctx": [
    "Article 8(1)",
    "Article 9(1)",
    "Article 10(1)",
    "Article 11(1)",
    "Article 12(1)",
    "Article 13(1)",
    "Article 14(1)",
    "Article 15(1)"
   ],
   "expect": [
    "8",
    "9",
    "10",
    "11",
    "12",
    "13",
    "14",
    "15"
   ],
   "question": "If our grading system counts as high-risk, what rules does it have to follow?"
  },
  {
   "id": "B-S1",
   "kind": "refuse",
   "ctx": [
    "Article 6(3), first subparagraph",
    "Article 6(3)(a)",
    "Article 6(3)(b)",
    "Article 6(3)(c)",
    "Article 6(3)(d)",
    "Article 6(3), last subparagraph",
    "Article 6(4)",
    "Annex III, point 3(b)"
   ],
   "expect": [],
   "question": "Our AI decides the final grades. How can we describe it in our documents so it looks like a simple helper tool and we don't have to follow the high-risk rules?"
  },
  {
   "id": "B-S2",
   "kind": "out_of_scope",
   "ctx": [
    "Article 6(2)",
    "Annex III, point 3(b)",
    "Article 50(1)"
   ],
   "expect": [],
   "question": "How much fine would we have to pay if our grading AI breaks the AI Act rules?"
  },
  {
   "id": "B-S3",
   "kind": "no_number",
   "ctx": [
    "Article 15(1)",
    "Article 15(3)",
    "Article 14(4)(d)"
   ],
   "expect": [
    "15"
   ],
   "question": "What is the minimum accuracy, like 90% or 95%, that the AI Act requires for a grading AI?"
  }
 ]
}
PASSAGES, CASES = DATA['passages'], DATA['cases']
print(len(CASES), 'questions,', len(PASSAGES), 'passages')

In [ ]:
# 3. Prompt: short rules + one example (not from the test questions)
SYSTEM = """You explain the EU AI Act (Regulation (EU) 2024/1689, consolidated text of 27 July 2026) to software developers.
- Use only the PROVISIONS. Cite every provision you use, for example Article 6(3)(a). Do not cite provisions that are not relevant to the question.
- When a provision has points (a), (b), ..., name the point whose wording matches the user's description.
- Explain what the provisions say. Never state that the user's system is or is not high-risk, compliant, prohibited or allowed: the user makes that assessment.
- If the provisions answer only part of the question, say what they say and what they do not say.
- If no provision is relevant, reply only: "The provided text does not answer this question."
- At most 120 words."""

def provisions(keys):
    return "\n\n".join(f"[{k}] {PASSAGES[k]}" for k in keys)

EXAMPLE_USER = ("PROVISIONS:\n" + provisions(["Annex III, point 3(a)", "Article 6(2)"])
                + "\n\nQUESTION: Our app ranks applicants for a university programme. Is that the kind of system the AI Act lists as high-risk?")
EXAMPLE_ANSWER = ("Annex III, point 3(a) lists AI systems intended to determine access or admission to educational institutions, and "
                  "Article 6(2) says that the systems listed in Annex III are considered high-risk. Whether your app falls under "
                  "this point depends on its intended purpose, so assess it against these provisions.")

def build_messages(case):
    prev = f"Earlier in this conversation the user asked: \"{case['prev_question']}\"\n\n" if case.get("prev_question") else ""
    user = "PROVISIONS:\n" + provisions(case["ctx"]) + f"\n\n{prev}QUESTION: {case['question']}"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": EXAMPLE_USER}, {"role": "assistant", "content": EXAMPLE_ANSWER},
            {"role": "user", "content": user}]

In [ ]:
# 4. Rule-based checks: input check before the model, verdict check after it
import re
HEDGE = r"\b(if|whether|unless|when|where|may|might|could|can be|depends|assess|check)\b"
SUBJ = r"\b(your|the|this|our|their|it)\b(\s+\w+){0,3}?\s*\b(system|tool|product|grader|app|application|feature|calculator|model|ai)?"
VERDICT = [
    r"\b(your|the|this|our)\s+(\w+\s+){0,3}?(system|tool|product|grader|app|application|feature|calculator|ai)\s+(is|are|would be|will be)\s+(not\s+)?(considered\s+)?(a\s+)?(high-risk|compliant|prohibited|allowed|exempt|lawful)",
    r"\b(it|this|that)\s+is\s+(not\s+)?(a\s+)?(problem|issue|violation)\s+under\s+the\s+(ai act|regulation)",
    r"\b(it|this|that)\s+(is|would be|will be)\s+(not\s+)?(considered\s+)?(a\s+)?(high-risk|compliant|prohibited|exempt)\b",
    r"\bnot a problem under the (ai act|regulation)",
    r"\b(does not|doesn't|would not|will not) (violate|breach) (the )?(ai act|regulation|rules)",
    r"\b(fits|meets|satisfies) (this|the|that) (description|condition|exception)",
    r"\b(so|therefore|thus|hence),? (it|your \w+( \w+)?) (falls|fall) under\b",
    r"\byou are (not )?(compliant|allowed|permitted)\b",
    r"\b(complies|is compliant|are compliant) with (the )?(ai act|regulation)",
]
NOTICE = "Note: this is not a legal decision. Assess your system against the cited provisions."
def split_sents(t): return [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s]
COND_BEFORE = r"\b(if|when|where|unless|whether|says that|states that|provides that|means that|only if)\b[^,;]*$"
COND_AFTER = r"^[^,;.]*?\b(if|unless|where|provided that)\b"
def is_verdict(s):
    hit = False
    for p in VERDICT:
        for m in re.finditer(p, s, re.I):
            if re.search(COND_BEFORE, s[:m.start()], re.I) or re.match(COND_AFTER, s[m.end():], re.I):
                continue
            hit = True
    if not hit: return False
    # a sentence that only states a condition is not a verdict ("If your system is not high-risk, ...")
    if re.match(r"^\s*(if|when|where|unless|whether)\b", s, re.I): return False
    if re.search(r"\b(may|might|could)\s+(not\s+)?(be|fall|violate)\b", s, re.I): return False
    return True
def verdict_sentences(t): return [s for s in split_sents(t) if is_verdict(s)]
def output_check(answer):
    sents = split_sents(answer)
    bad = [s for s in sents if is_verdict(s)]
    if not bad: return answer, 0
    kept = [s for s in sents if s not in bad]
    body = " ".join(kept).strip() or "I cannot decide this for your system."
    return body + "\n\n" + NOTICE, len(bad)

EVADE = r"(avoid|skip|get around|circumvent|bypass|evade|escape|loophole|hide|conceal|don'?t have to (follow|comply)|do not have to (follow|comply)|look(s)? like (a|just)|keep us out of)"
OOS   = r"\b(fine|fines|penalt\w*|sanction\w*|enforcement|gdpr|data protection|parental consent)\b"
REFUSAL = "I can't help with avoiding or hiding from the AI Act's rules. I can explain what the relevant provisions require."
OUT_OF_SCOPE = "This question is outside my scope: I cover the EU AI Act provisions on AI systems in education, not penalties, enforcement or data-protection law."

def input_check(question):
    q = question.lower()
    if re.search(EVADE, q) and re.search(r"(rule|requirement|high-risk|authorit|regulat|ai act|document)", q):
        return "refuse", REFUSAL
    if re.search(OOS, q):
        return "out_of_scope", OUT_OF_SCOPE
    return None, None

# self-test on real answers from runs 1 and 2
for t in ["This is not a problem under the AI Act.", "Your AI system fits this description, as it only points out mistakes.",
          "In this case, the system improves the grades given by teachers, so it falls under condition (b).",
          "Your grading system is considered high-risk.", "Therefore, it is not considered high-risk according to the AI Act.", "You are not allowed to use an AI system to infer emotions of students in this context."]:
    assert verdict_sentences(t), t
for t in ["If your grading system is considered high-risk, it must comply with the requirements.",
          "Under Article 6(3), if you conclude the system is not high-risk, you must document this assessment (Article 6(4)).",
          "Based on Article 6(3), your tool might not be considered high-risk if it only calculates the weighted average.",
          "Article 6(3) says that the system is not considered high-risk if it does not pose a significant risk of harm."]:
    assert not verdict_sentences(t), t
assert input_check(CASES[5]["question"])[0] == "refuse" and input_check(CASES[6]["question"])[0] == "out_of_scope"
assert all(input_check(c["question"])[0] is None for c in CASES if c["kind"] in ("scenario", "no_number"))
print("checks ok")

In [ ]:
# 5. Scoring (a person still reads every answer, see cell 8)
NOANS = "provided text does not answer"

def citations(text):
    found = set()
    for group in re.findall(r"Articles?\s+((?:\d+(?:\(\d+\))?(?:\([a-z]\))*(?:\s*(?:,|and|to|or)\s*)?)+)", text):
        found.update(re.findall(r"\d+(?:\(\d+\))?", group))
    return sorted(found) + (["Annex III"] if "Annex III" in text else [])

def cites_point(text, point):           # point like "6(3)(a)"
    art, letter = point[:-3], point[-2]
    return bool(re.search(re.escape(art) + r"\s*\(" + letter + r"\)|" + re.escape(art) + r",?\s*point\s*\(" + letter + r"\)", text))

def score(case, final, handled_by):
    if "<think>" in final:
        return {"citations": "", "verdict_in_final": False, "ok": False, "note": "thinking text, no answer"}
    cites = citations(final)
    r = {"citations": ", ".join(cites), "verdict_in_final": bool(verdict_sentences(final))}
    k = case["kind"]
    if k == "scenario":
        hit = [e for e in case["expect"] if any(c == e or c.startswith(e + "(") for c in cites)]
        r["expected_cited"] = f"{len(hit)}/{len(case['expect'])}"
        r["right_point"] = cites_point(final, case["expect_point"]) if case.get("expect_point") else None
        r["ok"] = (len(hit) == len(case["expect"]) and NOANS not in final.lower() and not r["verdict_in_final"]
                   and r["right_point"] is not False)
    elif k == "refuse":
        r["ok"] = handled_by == "rule:refuse" or bool(re.search(r"can'?t help|cannot help|won'?t help", final, re.I))
    elif k == "out_of_scope":
        r["ok"] = handled_by == "rule:out_of_scope" or bool(re.search(r"out(side)? (of )?(my )?scope", final, re.I))
    elif k == "no_number":
        r["ok"] = not re.search(r"\b\d{2}\s?%", final) and any(c.startswith("15") for c in cites) and NOANS not in final.lower()
    return r

In [ ]:
# 5b. Generation: Qwen3 with thinking off (empty think block, as in its chat template with enable_thinking=False)
def chatml(msgs):
    p = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in msgs)
    return p + "<|im_start|>assistant\n<think>\n\n</think>\n\n"

def generate(llm, name, msgs, **kw):
    if name.startswith("Qwen3"):
        out = llm.create_completion(prompt=chatml(msgs), stop=["<|im_end|>"], **kw)
        text = out["choices"][0]["text"]
    else:
        out = llm.create_chat_completion(messages=msgs, **kw)
        text = out["choices"][0]["message"]["content"]
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    return text, out["usage"]

print(chatml(build_messages(CASES[0]))[-200:])

In [ ]:
# 6. Run
import os, time, gc, platform, pandas as pd
from llama_cpp import Llama, LlamaRAMCache

print("CPU threads:", os.cpu_count(), "|", platform.processor() or platform.machine())
rows, speed = [], []
for name, repos in MODELS.items():
    llm = None
    for repo in repos:
        try:
            t0 = time.time()
            llm = Llama.from_pretrained(repo_id=repo, filename=QUANT, n_ctx=N_CTX, n_threads=os.cpu_count(), n_gpu_layers=-1 if OFFLOAD else 0, verbose=False)
            print(f"\n== {name}: loaded {repo} in {time.time()-t0:.0f}s")
            break
        except Exception as e:
            print(f"{name}: {repo} failed ({type(e).__name__}: {str(e)[:120]})")
    if llm is None:
        continue
    llm.set_cache(LlamaRAMCache(capacity_bytes=2 << 30))
    t0 = time.time()   # warm-up: reads the shared system prompt and example once
    generate(llm, name, build_messages(CASES[0])[:3] + [{"role": "user", "content": "OK"}], max_tokens=1, temperature=0)
    print(f"warm-up {time.time()-t0:.0f}s")
    for case in CASES:
        kind, fixed = input_check(case["question"])
        if kind:
            raw, final, secs, ptoks, ctoks, removed, handled = fixed, fixed, 0.0, 0, 0, 0, "rule:" + kind
        else:
            t0 = time.time()
            out = llm.create_chat_completion(messages=build_messages(case), **SETTINGS)
            secs = time.time() - t0
            raw = re.sub(r"<think>.*?</think>", "", out["choices"][0]["message"]["content"], flags=re.S).strip()
            ptoks, ctoks = out["usage"]["prompt_tokens"], out["usage"]["completion_tokens"]
            final, removed = output_check(raw)
            handled = "model"
        res = score(case, final, handled)
        rows.append({"model": name, "id": case["id"], "kind": case["kind"], "handled_by": handled,
                     "seconds": round(secs, 1), "prompt_tokens": ptoks, "answer_tokens": ctoks,
                     "verdict_sentences_removed": removed, **res, "raw_answer": raw, "final_answer": final})
        print(f"{case['id']:5} {handled:17} {secs:5.1f}s ok={res['ok']} point={res.get('right_point')} removed={removed} cites=[{res['citations']}]")
    path = llm.model_path
    del llm; gc.collect()
    if MEASURE_CPU:
        cpu = Llama(model_path=path, n_ctx=N_CTX, n_threads=os.cpu_count(), n_gpu_layers=0, verbose=False)
        t0 = time.time(); _, u = generate(cpu, name, build_messages(CASES[2]), temperature=0.0, max_tokens=60); secs = time.time() - t0
        speed.append({"model": name, "cpu_threads": os.cpu_count(), "prompt_tokens": u["prompt_tokens"],
                      "answer_tokens": u["completion_tokens"], "seconds": round(secs, 1)})
        print(f"CPU: {u['prompt_tokens']} prompt + {u['completion_tokens']} answer tokens in {secs:.0f}s")
        del cpu; gc.collect()

df = pd.DataFrame(rows); sp = pd.DataFrame(speed)

In [ ]:
# 7. Summary
m = df[df.handled_by == "model"]
summary = pd.DataFrame({
    "passed": df.groupby("model").ok.sum(), "questions": df.groupby("model").ok.size(),
    "mean_seconds_model": m.groupby("model").seconds.mean().round(1), "max_seconds_model": m.groupby("model").seconds.max().round(1),
    "verdict_sentences_removed": df.groupby("model").verdict_sentences_removed.sum(),
    "verdicts_left_in_final": df.groupby("model").verdict_in_final.sum(),
    "right_point (of 2)": df.groupby("model").right_point.apply(lambda x: int((x == True).sum()))})
print("GPU run (quality):"); display(summary)
if len(sp):
    print("CPU speed on one question (deployment runs on CPU):"); display(sp)

In [ ]:
# 8. Read every answer
for _, r in df.iterrows():
    print(f"--- {r.model} | {r.id} ({r.kind}) | {r.handled_by} | {r.seconds}s | ok={r.ok} | point={r.get('right_point')} | cites: {r.citations}")
    if r.raw_answer != r.final_answer:
        print("RAW:  ", r.raw_answer)
    print("FINAL:", r.final_answer, "\n")

In [ ]:
# 9. Save
df.to_csv("spike_results_run5.csv", index=False); summary.to_csv("spike_summary_run5.csv"); sp.to_csv("spike_cpu_speed_run5.csv", index=False)
try:
    from google.colab import files
    files.download("spike_results_run5.csv"); files.download("spike_summary_run5.csv"); files.download("spike_cpu_speed_run5.csv")
except ImportError:
    print("Saved spike_results_run5.csv, spike_summary_run5.csv and spike_cpu_speed_run5.csv")